# Test Cory's code 2 spheres with spherical harmonics for scattered field

We solve for the scattered field
$$
\begin{aligned}
&\Delta u + k^2 u = 0 &\quad \text{in } &E:= \mathbb{R}^3 \setminus \bigcup_{i = 1}^2 \bar{D}_i \\
&\partial_{n_i} u =  - {\partial_{n_i}u^{\text{inc}} } &\quad \text{on } &B_i := \partial D_i, \, i =1, 2 \\
\end{aligned}
$$

which admits the representation 
$$u(x) =  \sum \limits_{i=1}^2 \displaystyle \int_{B_i} \partial_{n_i} {G}(x,y_i) {u_i(y_i)} d\sigma_{y_i} + \displaystyle \int_{B_i} {G}(x,y_i){\partial_{n_i} u^{\text{inc}}(y_i)} d\sigma_{y_i} $$

Assuming the spherical harmonic expansion
${u_i}= \displaystyle \sum_{n = 0 }^{\infty} C_{n}^{(i)} Y_{n}^0, \quad i = 1,2$
we solve the BIE system

$$
\begin{bmatrix}  
\tfrac{1}{2}I - \textcolor{orange}{\mathcal{D}_{11}} & - \textcolor{orange}{\mathcal{D}_{21}}  \\
  - \textcolor{orange}{\mathcal{D}_{12}}& \tfrac{1}{2}I- \textcolor{orange}{\mathcal{D}_{22}}  \\
\end{bmatrix}\begin{bmatrix} \textcolor{Aquamarine}{u_1} \\ \textcolor{Aquamarine}{u_2} \end{bmatrix} = 
\begin{bmatrix}  
\textcolor{orange}{\mathcal{S}_{11}} & \textcolor{orange}{\mathcal{S}_{21}} \\
 \textcolor{orange}{\mathcal{S}_{12}} &  \textcolor{orange}{\mathcal{S}_{22}}  \\
\end{bmatrix}
\begin{bmatrix} \textcolor{red}{\partial_{n_1}u_1^{\text{inc}}}  \\ \textcolor{red}{\partial_{n_N}u_2^{\text{inc}}} \end{bmatrix} = \begin{bmatrix}  
\tfrac{1}{2}I + \textcolor{orange}{\mathcal{D}_{11}} &  \textcolor{orange}{\mathcal{D}_{21}}  \\
\textcolor{orange}{\mathcal{D}_{12}}& \tfrac{1}{2}I+ \textcolor{orange}{\mathcal{D}_{22}}  \\
\end{bmatrix}\begin{bmatrix} \textcolor{red}{u_1^{\text{inc}}}  \\ \textcolor{red}{u_2^{\text{inc}}} \end{bmatrix}
$$

with 
$$\textcolor{orange}{\mathcal{S}_{ij}} [\textcolor{red}{\partial_n u_i^{\text{inc}}}] := \displaystyle \int_{B_i} \textcolor{orange}{G}(x_j,y_i) \textcolor{red}{\partial_{n_i} u_i^{\text{inc}}(y_i)} d\sigma_{y_i}, \quad \textcolor{orange}{\mathcal{D}_{ij}} [\textcolor{Aquamarine}{u_i}] := \displaystyle \int_{B_i} \partial_{n_i} \textcolor{orange}{G}(x_j,y_i) \textcolor{Aquamarine}{u_i(y_i)} d\sigma_{y_i}$$

(notation probably reversed in Cory's quals ?)

The code allows to also compute for the TOTAL field: 
$$
\begin{aligned}
&\Delta u + k^2 u = 0 &\quad \text{in } &E:= \mathbb{R}^3 \setminus \bigcup_{i = 1}^2 \bar{D}_i \\
&\partial_{n_i} u = 0 &\quad \text{on } &B_i := \partial D_i, \, i =1, 2 \\
\end{aligned}
$$
then 
$$u(x) = u^{\text{inc}} + \sum \limits_{i=1}^2 \displaystyle \int_{B_i} \partial_{n_i} {G}(x,y_i) {u_i(y_i)} d\sigma_{y_i}  $$
and we solve 
$$
\begin{bmatrix}  
\tfrac{1}{2}I - \textcolor{orange}{\mathcal{D}_{11}} & - \textcolor{orange}{\mathcal{D}_{21}}  \\
  - \textcolor{orange}{\mathcal{D}_{12}}& \tfrac{1}{2}I- \textcolor{orange}{\mathcal{D}_{22}}  \\
\end{bmatrix}\begin{bmatrix} \textcolor{Aquamarine}{u_1} \\ \textcolor{Aquamarine}{u_2} \end{bmatrix} = 
\begin{bmatrix} \textcolor{red}{u_1^{\text{inc}}}  \\ \textcolor{red}{u_2^{\text{inc}}} \end{bmatrix}
$$

In [1]:
import matplotlib as mpl
import numpy as np
import scipy.special as sp
import matplotlib.pyplot as plt

def colorFader(c1,c2,mix=0): #fade (linear interpolate) from color c1 (at mix=0) to c2 (mix=1)
    c1=np.array(mpl.colors.to_rgb(c1))
    c2=np.array(mpl.colors.to_rgb(c2))
    return mpl.colors.to_hex((1-mix)*c1 + mix*c2)

c1='#00008B' #blue
c2='#ADD8E6' #light blue

In [2]:
def compute_inc_coeff(a1, a2, k, N):
    B_n1 = np.zeros( N, dtype = 'complex' )
    B_n2 = np.zeros( N, dtype = 'complex' )
    for n in range(N): 
        B_n1[n] = 4 * np.pi * (1j)**n * sp.spherical_jn( n, k * a1, derivative = False )
        B_n2[n] = 4 * np.pi * (1j)**n * sp.spherical_jn( n, k * a2, derivative = False )
    return B_n1, B_n2

In [3]:
def compute_blocks(a1, a2, k, N, epsVec):
    '''
    Compute spherical harmonic coefficients while solving a sound-hard acoustic scattering problem by 2 spheres
    We use the 2 formulations given above
    '''
    Coeffs01 = np.zeros( (len(epsVec), N), dtype = 'complex' )
    Coeffs02 = np.zeros( (len(epsVec), N), dtype = 'complex' )
    Coeffs1 = np.zeros( (len(epsVec), N), dtype = 'complex' )
    Coeffs2 = np.zeros( (len(epsVec), N), dtype = 'complex' )
    Coeffs3 = np.zeros( (len(epsVec), N), dtype = 'complex' )
    Coeffs4 = np.zeros( (len(epsVec), N), dtype = 'complex' )

    # compute the Gauss-Legendre quadrature rule
    μ, wt = np.polynomial.legendre.leggauss( N )
    s = np.arccos(μ)
    # compute a vector of indices
    nvec = np.arange( N )
    # compute Legendre polynomials
    Pn = np.zeros( (N, N) )
    for n in nvec:
        #normalized polynomial NORMALIZATION IS INCLUDED
        Pn[:,n] = np.sqrt( ( 2.0 * n + 1 ) / ( 4.0 * np.pi ) ) * sp.eval_legendre( n, μ )
    #----------------------------------
    # BLOCK 11
    #compute spherical Bessel functions and their derivatives 
    jn1  = sp.spherical_jn( nvec, k * a1, derivative = False )
    yn1  = sp.spherical_yn( nvec, k * a1, derivative = False )
    Djn1 = sp.spherical_jn( nvec, k * a1, derivative = True )
    Dyn1  = sp.spherical_yn( nvec, k * a1, derivative = True )

    # compute spherical Hankel function of the first kind and its derivative
    h1n1  = jn1 + 1j * yn1
    Dh1n1 = Djn1 + 1j * Dyn1

    # compute the diagonal blocks
    λ1 = 1.0 - 1j * ( k * a1 ) ** 2 * Djn1 * h1n1 #* h1n1 #ADD h1 to mimic natural expansion ?
    d11 = 1j * (k * a1) * jn1 * h1n1
    λ1d = 1.0 + 1j * ( k * a1 ) ** 2 * jn1 * Dh1n1 # check notes on notations for exterior 
    #----------------------------------
    # BLOCK 22 
    #compute spherical Bessel functions and their derivatives
    jn2  = sp.spherical_jn( nvec, k * a2, derivative = False )
    yn2  = sp.spherical_yn( nvec, k * a2, derivative = False )
    Djn2 = sp.spherical_jn( nvec, k * a2, derivative = True )
    Dyn2 = sp.spherical_yn( nvec, k * a2, derivative = True )

    # compute spherical Hankel function of the first kind and its derivative
    h1n2  = jn2 + 1j * yn2
    Dh1n2  = Djn2 + 1j * Dyn2

    # compute the diagonal blocks
    λ2 = 1.0 - 1j * ( k * a2 ) ** 2 * Djn2 * h1n2 #* h1n2 #ADD h1 to mimic natural expansion ?
    d22 = 1j * (k * a2) * jn2 * h1n2
    λ2d = 1.0 + 1j * ( k * a2 ) ** 2 * jn2 * Dh1n2
    #----------------------------------
    # Off diagonal blocks
    for i, val in enumerate(epsVec):   
        Δz1     = 0
        Δz2     = a1 + a2 + val
        # set location of sphere 1 and sphere 2
        x1 = 0.0
        y1 = 0.0
        z1 = Δz1

        x2 = 0.0
        y2 = 0.0
        z2 = Δz2

        # compute the (1,2) off-diagonal block matrix (scattering from sphere 2 to sphere 1)
        x11 = x1 + a1 * np.sin(s)
        y11 = y1
        z11 = z1 + a1 * np.cos(s)

        # compute the (2,1) off-diagonal block matrix (scattering from sphere 1 to sphere 2)
        x22 = x2 + a2 * np.sin(s)
        y22 = y2
        z22 = z2 + a2 * np.cos(s)
        
        #coupling 1-2 (denoted 21 in matrix form)
        R21 = np.sqrt( ( x11 - x2 ) ** 2 + ( y11 - y2 ) ** 2 + ( z11 - z2 ) ** 2 )
        S21 = ( z11 - z2 ) / R21
        #coupline 2-1 (denoted 12 in matrix form)
        R12 = np.sqrt( ( x22 - x1 ) ** 2 + ( y22 - y1 ) ** 2 + ( z22 - z1 ) ** 2 )
        S12 = ( z22 - z1 ) / R12
        
        A12 = np.zeros( (N,N), dtype = 'complex' )
        B12 = np.zeros( (N,N), dtype = 'complex' )
        A21 = np.zeros( (N,N), dtype = 'complex' )
        B21 = np.zeros( (N,N), dtype = 'complex' )
        C12 = np.zeros( (N,N), dtype = 'complex' )
        C21 = np.zeros( (N,N), dtype = 'complex' )

        for n in nvec:
            P21 = sp.eval_legendre( n, S21 )
            j21 = sp.spherical_jn( n, k * R21, derivative = False )
            y21 = sp.spherical_yn( n, k * R21, derivative = False )
            h21 = j21 + 1j * y21
            dj21 = sp.spherical_jn( n, k * R21, derivative = True )
            dy21 = sp.spherical_yn( n, k * R21, derivative = True )
            dh21 = dj21 + 1j * dy21
            
            P12 = sp.eval_legendre( n, S12 )
            j12 = sp.spherical_jn( n, k * R12, derivative = False )
            y12 = sp.spherical_yn( n, k * R12, derivative = False )
            h12 = j12 + 1j * y12
            dj12 = sp.spherical_jn( n, k * R12, derivative = True )
            dy12 = sp.spherical_yn( n, k * R12, derivative = True )
            dh12 = dj12 + 1j * dy12

            A12[:,n] = 1j * ( k * a1 ) ** 2 * Djn1[n] * h12 * P12 
            B12[:,n] = 1j *  (k * a1) * jn1[n] * h12 * P12 
            C12[:,n] = 1j * ( k * a1 ) ** 2 * jn1[n] * dh12 * P12 
            A21[:,n] = 1j * ( k * a2 ) ** 2 * Djn2[n] * h21 * P21 
            B21[:,n] = 1j *  (k * a2) * jn2[n] * h21 * P21 
            C21[:,n] = 1j * ( k * a2 ) ** 2 * jn2[n] * dh21 * P21 

        # projection onto spherical harmonics (2 pi from the theta invariance)
        A12 =  2.0 * np.pi * Pn.T @ np.diag( wt ) @ A12
        B12 =  2.0 * np.pi * Pn.T @ np.diag( wt ) @ B12
        A21 =  2.0 * np.pi * Pn.T @ np.diag( wt ) @ A21
        B21 =  2.0 * np.pi * Pn.T @ np.diag( wt ) @ B21
        C12 =  2.0 * np.pi * Pn.T @ np.diag( wt ) @ C12
        C21 =  2.0 * np.pi * Pn.T @ np.diag( wt ) @ C21
        # solve the block system
        #NOTE: all blocks have an extra a1, a2 as we do not integrate on a unit sphere
        Ablock = np.block( [ [ np.diag(λ1), -A21 ], [ -A12, np.diag(λ2) ] ] )
    
        # CHOICE ONE projection onto spherical harmonics  PLANE WAVE   TOTAL FIELD    
        b01 = 2.0 * np.pi * Pn.T @ np.diag( wt ) @ np.exp( 1j*k * ( z1 + a1 * μ ) )
        b02 = 2.0 * np.pi * Pn.T @ np.diag( wt ) @ np.exp( 1j*k * ( z2 + a2 * μ ) )

        bblock0 = np.block( [ b01, b02 ] )
        c0 = np.linalg.solve( Ablock, bblock0)
        
        # CHOICE TWO projection onto spherical harmonics  PLANE WAVE SCATTERED FIELD
        b11 = (1j * k * (μ) * np.exp( 1j * k * ( z1 + a1 * μ ) ))
        b22 = (1j * k * (μ) * np.exp( 1j * k * ( z2 + a2 * μ ) ))
        b21 = (1j * k * (S21) * np.exp( 1j * k * ( z2 + a2 * μ ) ))
        b12 = (1j * k * (S12) * np.exp( 1j * k * ( z1 + a1 * μ ) ))
        b1 = np.diag(d11) @ b11 + B21 @ b21 #careful order, here consistent with choice of notations
        b2 = B12 @ b12 + np.diag(d22) @ b22

        bblock1 = np.block( [ b1, b2 ] )
        c1 = np.linalg.solve( Ablock, bblock1)

        # CHOICE THREE projection onto spherical harmonics  PLANE WAVE SCATTERED FIELD modified
        b3 = np.exp( 1j * k * ( z1 + a1 * μ ) )
        b4 = np.exp( 1j * k * ( z2 + a2 * μ ) )
        bblock2 = np.block( [ np.diag(λ1d)@b3 + C21@b4, C12@b3 + np.diag(λ2d)@b4 ] )
        c2 = np.linalg.solve( Ablock, bblock2)
    
        Coeffs01[i,:] = c0[:N]
        Coeffs02[i,:] = c0[N:]
        Coeffs1[i,:] = c1[:N]
        Coeffs2[i,:] = c1[N:]
        Coeffs3[i,:] = c2[:N]
        Coeffs4[i,:] = c2[N:]
    return Coeffs01, Coeffs02, Coeffs1, Coeffs2, Coeffs3, Coeffs4
    

In [4]:
epsVec  = np.array([1e-10, 1e-9, 1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1])

from ipywidgets import interact
@interact
def interact_plot(k=(0.5, 2, 0.1), N =(10, 100, 10)):
    fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6))= plt.subplots(3, 2, figsize = (12, 16))
    ax1.grid(True)
    ax2.grid(True)
    ax1.set_xlabel('N')
    ax2.set_xlabel('N')
    ax1.set_ylim((1e-16, 1e2))
    ax2.set_ylim((1e-16, 1e2))
    ax1.set_xlim((0, 1e2))
    ax2.set_xlim((0, 1e2))
    ax1.set_title(f'$C^{1}$ total')
    ax2.set_title(f'$C^{2}$ total')
    ax3.grid(True)
    ax4.grid(True)
    ax3.set_xlabel('N')
    ax4.set_xlabel('N')
    ax3.set_ylim((1e-16, 1e2))
    ax4.set_ylim((1e-16, 1e2))
    ax3.set_xlim((0, 1e2))
    ax4.set_xlim((0, 1e2))
    ax3.set_title(f'$C^{1}$ scattered')
    ax4.set_title(f'$C^{2}$ scattered')
    ax5.grid(True)
    ax6.grid(True)
    ax5.set_xlabel('N')
    ax6.set_xlabel('N')
    ax5.set_ylim((1e-16, 1e2))
    ax6.set_ylim((1e-16, 1e2))
    ax5.set_xlim((0, 1e2))
    ax6.set_xlim((0, 1e2))
    ax5.set_title(f'$C^{1}$ scattered modified')
    ax6.set_title(f'$C^{2}$ scattered modified')
    a1      = 1/k
    a2      = 1/k
    # Allocate storage for coefficients
    Coeffs01, Coeffs02, Coeffs1, Coeffs2, Coeffs3, Coeffs4 = compute_blocks(a1, a2, k, N, epsVec)
    # Compute plane wave spherical harmonic coefficients    
    B_n1, B_n2 = compute_inc_coeff(a1, a2, k, N)
    for i, val in enumerate(epsVec):
        ax1.semilogy(np.arange(N), np.abs(Coeffs01[i,:]), '.', color=colorFader(c1,c2,i/len(epsVec)), label = f'$\epsilon$ = {val}') 
        ax2.semilogy(np.arange(N), np.abs(Coeffs02[i,:]), '.', color=colorFader(c1,c2,i/len(epsVec)), label = f'$\epsilon$ = {val}') 
        ax3.semilogy(np.arange(N), np.abs(Coeffs1[i,:]), '.', color=colorFader(c1,c2,i/len(epsVec)), label = f'$\epsilon$ = {val}') 
        ax4.semilogy(np.arange(N), np.abs(Coeffs2[i,:]), '.', color=colorFader(c1,c2,i/len(epsVec)), label = f'$\epsilon$ = {val}') 
        ax5.semilogy(np.arange(N), np.abs(Coeffs3[i,:]), '.', color=colorFader(c1,c2,i/len(epsVec)), label = f'$\epsilon$ = {val}') 
        ax6.semilogy(np.arange(N), np.abs(Coeffs4[i,:]), '.', color=colorFader(c1,c2,i/len(epsVec)), label = f'$\epsilon$ = {val}') 
    ax1.semilogy(np.arange(N), np.abs(B_n1), '.', color='C1', label = f'$B_n$')
    ax2.semilogy(np.arange(N), np.abs(B_n2), '.', color='C1', label = f'$B_n$')
    ax2.legend(loc = 'best', bbox_to_anchor=(0.9, 0.5, 0.5, 0.5))
    #ax4.legend(loc = 'best', bbox_to_anchor=(0.9, 0.5, 0.5, 0.5))

interactive(children=(FloatSlider(value=1.2000000000000002, description='k', max=2.0, min=0.5), IntSlider(valu…